In [0]:
# Connect to Azure resources
spark.sql("USE CATALOG dbw_fleet_telemetry_dev")

storage_account   = "stfleettelemetryalvin"
eh_namespace      = "fleet-telemetry-streaming"
eh_name           = "fleet-telemetry-v2"
eh_connection_str = "YOUR_EVENT_HUB_CONNECTION_STRING"
storage_key       = "YOUR_STORAGE_ACCOUNT_KEY"

spark.conf.set(
    f"fs.azure.account.key.{storage_account}.dfs.core.windows.net",
    storage_key
)

CHECKPOINT_PATH = f"abfss://bronze@{storage_account}.dfs.core.windows.net/_checkpoints/bronze"

print(f"Catalog: {spark.catalog.currentCatalog()}")
print("Connected")

Catalog: dbw_fleet_telemetry_dev
Connected


In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, FloatType

telemetry_schema = StructType([
    StructField("vehicle_id",    StringType(), False),
    StructField("timestamp",     StringType(), False),
    StructField("lat",           DoubleType(), True),
    StructField("lon",           DoubleType(), True),
    StructField("speed_kmh",     FloatType(),  True),
    StructField("fuel_pct",      FloatType(),  True),
    StructField("engine_temp_c", FloatType(),  True),
    StructField("odometer_km",   FloatType(),  True),
    StructField("event_type",    StringType(), True),
    StructField("route_id",      StringType(), True),
])

print("Schema defined")

Schema defined


In [0]:
from pyspark.sql import functions as F
KAFKA_BOOTSTRAP = f"{eh_namespace}.servicebus.windows.net:9093"

SASL_CONFIG = (
    'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required '
    f'username="$ConnectionString" '
    f'password="{eh_connection_str}";'
)

raw_stream = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers",  KAFKA_BOOTSTRAP)
    .option("kafka.security.protocol",  "SASL_SSL")
    .option("kafka.sasl.mechanism",     "PLAIN")
    .option("kafka.sasl.jaas.config",   SASL_CONFIG)
    .option("subscribe",                eh_name)
    .option("startingOffsets",          "earliest")
    .option("failOnDataLoss",           "false")
    .load()
)

print("Stream reader defined")
raw_stream.printSchema()

Stream reader defined
root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



In [0]:
from pyspark.sql import functions as F

parsed_stream = (
    raw_stream
    .withColumn("raw_json", F.col("value").cast("string"))
    .withColumn("data", F.from_json(F.col("raw_json"), telemetry_schema))
    .select(
        "data.*",
        F.col("partition").alias("kafka_partition"),
        F.col("offset").alias("kafka_offset"),
        F.col("timestamp").alias("kafka_enqueue_time"),
    )
    .withColumn("event_time", F.to_timestamp("timestamp", "yyyy-MM-dd'T'HH:mm:ss.SSSSSSXXX"))
    .drop("timestamp")
    .withColumn("ingestion_time", F.current_timestamp())
    .filter(F.col("vehicle_id").isNotNull())
    .withColumn("severity",
        F.when(F.col("event_type") == "engine_overheat", "CRITICAL")
        .when(F.col("event_type") == "fuel_critical",    "HIGH")
        .when(F.col("event_type") == "engine_vibration", "HIGH")
        .when(F.col("event_type") == "battery_failure",  "HIGH")
        .when(F.col("event_type") == "speed_breach",     "MEDIUM")
        .when(F.col("event_type") == "harsh_braking",    "MEDIUM")
        .when(F.col("event_type") == "route_deviation",  "MEDIUM")
        .when(F.col("event_type") == "gps_signal_loss",  "LOW")
        .otherwise("NORMAL")
    )
    .withColumnRenamed("event_type", "alert_type")
)

print("Stream parsed")
parsed_stream.printSchema()

Stream parsed
root
 |-- vehicle_id: string (nullable = true)
 |-- lat: double (nullable = true)
 |-- lon: double (nullable = true)
 |-- speed_kmh: float (nullable = true)
 |-- fuel_pct: float (nullable = true)
 |-- engine_temp_c: float (nullable = true)
 |-- odometer_km: float (nullable = true)
 |-- alert_type: string (nullable = true)
 |-- route_id: string (nullable = true)
 |-- kafka_partition: integer (nullable = true)
 |-- kafka_offset: long (nullable = true)
 |-- kafka_enqueue_time: timestamp (nullable = true)
 |-- event_time: timestamp (nullable = true)
 |-- ingestion_time: timestamp (nullable = false)
 |-- severity: string (nullable = false)



In [0]:
bronze_query = (
    parsed_stream
    .writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_PATH)
    .option("mergeSchema", "false")
    .trigger(processingTime="30 seconds")
    .toTable("bronze.raw_telemetry")
)

In [0]:
spark.sql("""
    SELECT alert_type, severity, COUNT(*) as count
    FROM bronze.raw_telemetry
    GROUP BY alert_type, severity
    ORDER BY count DESC
""").show()

+---------------+--------+------+
|     alert_type|severity| count|
+---------------+--------+------+
|         normal|  NORMAL|114229|
|   speed_breach|  MEDIUM| 19635|
|engine_overheat|CRITICAL| 14566|
|  fuel_critical|    HIGH|  9741|
|route_deviation|  MEDIUM|  4859|
+---------------+--------+------+

